# NB26 · Type Hints

`advanced/` · notebook 26 of 32 · 34 questions + a mini-project

Type hints are notes about what kind of value a name holds: `price: float`,
`def total(items: list[float]) -> float`. Python stores them but **never
checks them** while your program runs. Tools like mypy and pyright read
them before you run anything and point out mistakes your tests missed.

That makes this notebook a little unusual. Every question still prints
something real, because hints *are* available at runtime as plain objects:
`__annotations__`, `typing.get_type_hints()`, `get_origin()`/`get_args()`,
a `type` alias's `__value__`, a generic's `__type_params__`, and
`isinstance()` checks against a `@runtime_checkable` Protocol. You will also
write small runtime validators that read hints and check real data.

Questions tagged `[3.13+]` or `[3.14+]` need that Python version or newer.
Python 3.14 changed one big thing: annotations are now evaluated lazily,
only when something asks for them.

## Prerequisites

Before you start, you should be comfortable with:

- [ ] NB25: metaprogramming
- [ ] NB21: dataclasses

## What you'll practise

- Annotating variables, parameters and return values
- Built-in generics (list[int]); X | None; Callable; collections.abc types
- Generics: TypeVar, and the PEP 695 syntax def f[T](x: T) and class Box[T]
- type aliases with the type statement (3.12)
- Protocol, TypedDict, Literal, Final, ClassVar, Self, overload, NewType, cast
- Hints at runtime: get_type_hints, deferred annotations in 3.14
- What mypy/pyright catch that tests don't
- Newer typing: @override (3.12), TypeIs and ReadOnly (3.13), type-parameter defaults (3.13)

## How to use this notebook

1. Read the question and write your code in the cell beneath it.
2. Run the cell, then open **Expected output** and compare.
3. Stuck? Open **Hint** first. Open **Solution** only after a real attempt.
4. Once it works, open **Senior dev solution** to see how an experienced developer would write it, and why.

The **Solution** only uses what this notebook and the ones before it teach. The **Senior dev solution** sometimes uses later topics on purpose, and says which notebook covers them.

| Type | What you do |
|---|---|
| **Write** | Write code that prints the answer |
| **Predict** | Read the code and predict exactly what it prints |
| **Error** | Predict which error the code raises |
| **Fix** | The code has a bug. Find it and fix it |
| **Refactor** | The code works. Make it cleaner without changing its output |

| Level | Meaning |
|---|---|
| L1 | Extremely Easy: one idea, a line or two |
| L2 | Easy: one idea with a small twist |
| L3 | Intermediate: two or three ideas combined |
| L4 | Somewhat Difficult: needs care to get exactly right |
| L5 | Hard: several ideas and a trap or two |

---

## L1 · Extremely Easy

### NB26-Q01 · L1 · Annotating functions · Write

Write a function `area(width, height)` that returns `width * height`.
Annotate both parameters and the return value as `float`. Then print
`area(2.5, 4.0)` and, on the next line, the function's `__annotations__`.

<details><summary><b>Hint</b></summary>

A parameter hint goes after a colon (`width: float`). The return hint goes after `->`, before the final colon of the `def` line.

</details>

<details><summary><b>Expected output</b></summary>

```text
10.0
{'width': <class 'float'>, 'height': <class 'float'>, 'return': <class 'float'>}
```

</details>

<details><summary><b>Solution</b></summary>

```python
def area(width: float, height: float) -> float:
    return width * height

print(area(2.5, 4.0))
print(area.__annotations__)
```

Hints are stored on the function in a dict called `__annotations__`. The keys are parameter names, plus `'return'` for the `->` hint. The values are the real objects you wrote, here the `float` class itself.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def area(width: float, height: float) -> float:
    """Area of a rectangle, in the same units squared."""
    return width * height

print(area(2.5, 4.0))
print(area.__annotations__)
```

Hints say *what type*, a docstring says *what it means*. Seniors write both on public functions. `float` also accepts an `int` for type checkers, so `area(2, 3)` is fine to mypy.

</details>

In [ ]:
# your code here


### NB26-Q02 · L1 · Hints are not enforced · Predict

The hint says `int`, but the call passes a string. Predict the output.

**What does this print?**

```python
def double(n: int) -> int:
    return n * 2

print(double("ab"))
```

<details><summary><b>Hint</b></summary>

Does Python look at the hint when you call the function?

</details>

<details><summary><b>Expected output</b></summary>

```text
abab
```

</details>

<details><summary><b>Solution</b></summary>

It prints `abab`. Python **never enforces hints at runtime**. The function runs exactly as if the hints weren't there, and `"ab" * 2` repeats the string. A type checker such as mypy would flag `double("ab")` before you ran it, and that is the whole point of hints.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def double(n: int) -> int:
    if not isinstance(n, int):
        raise TypeError("n must be an int")
    return n * 2

print(double(21))
```

Hints catch mistakes in *your* code before it runs. Data from outside (files, users, APIs) still needs a real runtime check like `isinstance()`, because a hint can't stop bad input arriving.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB26-Q03 · L1 · Optional values · Write

Write `find_price(name)` that returns the price from `prices`, or `None`
when the item isn't there. Annotate `name` as `str` and the return value
as "a float or None" using the `|` syntax. Print
`find_price("tea")`, `find_price("cake")`, then on a new line the return hint.
```text
2.5 None
float | None
```

**Starting code (already in the cell below):**

```python
prices = {"tea": 2.5, "coffee": 3.0}
```

<details><summary><b>Hint</b></summary>

`dict.get()` already returns `None` for a missing key. Write the return type as `float | None`. The return hint lives under the `'return'` key.

</details>

<details><summary><b>Expected output</b></summary>

```text
2.5 None
float | None
```

</details>

<details><summary><b>Solution</b></summary>

```python
def find_price(name: str) -> float | None:
    return prices.get(name)

print(find_price("tea"), find_price("cake"))
print(find_price.__annotations__["return"])
```

`float | None` means "either a float or `None`". It is a real object (a union) and prints as you wrote it. Spelling out `| None` tells every caller they must handle the missing case, and mypy will complain if they do maths on the result without checking for `None` first.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def find_price(name: str) -> float | None:
    return prices.get(name)

price = find_price("cake")
print(find_price("tea"), price)
print(find_price.__annotations__["return"])
```

Older code writes `Optional[float]` from `typing`. It means the same thing; `float | None` (3.10+) is shorter and is what seniors write today.

</details>

In [ ]:
prices = {"tea": 2.5, "coffee": 3.0}

# your code here


---

## L2 · Easy

### NB26-Q04 · L2 · get_type_hints · Write

The function below works but has no hints. Rewrite it with hints:
`prices` is a list of floats, `tax_rate` is a float, and it returns a float.
Then print `total([10.0, 5.0], 0.2)` and, on the next line, the result of
`get_type_hints(total)` (import it from `typing`).

**Starting code (already in the cell below):**

```python
def total(prices, tax_rate):
    return sum(prices) * (1 + tax_rate)
```

<details><summary><b>Hint</b></summary>

A list of floats is written `list[float]`. `get_type_hints` takes the function itself, not a call to it.

</details>

<details><summary><b>Expected output</b></summary>

```text
18.0
{'prices': list[float], 'tax_rate': <class 'float'>, 'return': <class 'float'>}
```

</details>

<details><summary><b>Solution</b></summary>

```python
from typing import get_type_hints

def total(prices: list[float], tax_rate: float) -> float:
    return sum(prices) * (1 + tax_rate)

print(total([10.0, 5.0], 0.2))
print(get_type_hints(total))
```

Redefining `total` replaces the old version. `list[float]` uses the built-in `list` as a *generic*: the part in square brackets says what the list holds. `get_type_hints()` returns the same information as `__annotations__`, but it also resolves hints written as strings and handles inheritance for classes, so it is the safer way to read hints.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from typing import get_type_hints

def total(prices: list[float], tax_rate: float) -> float:
    return sum(prices) * (1 + tax_rate)

print(total([10.0, 5.0], 0.2))
print(get_type_hints(total))
```

Same code. For a parameter you only loop over, seniors often accept the wider `Iterable[float]` from `collections.abc`, so tuples and generators work too. Accept wide types, return precise ones.

</details>

In [ ]:
def total(prices, tax_rate):
    return sum(prices) * (1 + tax_rate)

# your code here


### NB26-Q05 · L2 · Generics at runtime · Error

A first attempt at checking that a value is a list of ints. Which error does this raise?

**Which error does this raise?**

```python
scores = [70, 85, 92]
print(isinstance(scores, list[int]))
```

<details><summary><b>Hint</b></summary>

Could `isinstance()` check every item for you? Would it want to?

</details>

<details><summary><b>Expected output</b></summary>

```text
TypeError
```

</details>

<details><summary><b>Solution</b></summary>

`TypeError`: `isinstance()` refuses a *parameterised generic* like `list[int]`. Checking it would mean looking at every item, which `isinstance()` never does. Check the container with `isinstance(scores, list)` and the items yourself, e.g. `all(isinstance(s, int) for s in scores)`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
scores = [70, 85, 92]
print(isinstance(scores, list) and all(isinstance(s, int) for s in scores))
```

Split the check in two: the container type, then the items. Validator libraries (pydantic, msgspec) do exactly this, driven by the hints.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB26-Q06 · L2 · Callable · Write

Write `apply_twice(func, value)` that returns `func(func(value))`.
Annotate `func` as a callable taking one `int` and returning an `int`
(import `Callable` from `collections.abc`), `value` as `int`, and the
return as `int`. Print `apply_twice(add_ten, 5)`, then on a new line
the hint for `func` from `apply_twice.__annotations__`.
```text
25
collections.abc.Callable[[int], int]
```

**Starting code (already in the cell below):**

```python
def add_ten(n: int) -> int:
    return n + 10
```

<details><summary><b>Hint</b></summary>

The shape is `Callable[[argument types], return type]`. The argument types go inside their own square brackets.

</details>

<details><summary><b>Expected output</b></summary>

```text
25
collections.abc.Callable[[int], int]
```

</details>

<details><summary><b>Solution</b></summary>

```python
from collections.abc import Callable

def apply_twice(func: Callable[[int], int], value: int) -> int:
    return func(func(value))

print(apply_twice(add_ten, 5))
print(apply_twice.__annotations__["func"])
```

`Callable[[int], int]` describes any function (or object with `__call__`) that takes one `int` and returns an `int`. The inner list holds the parameter types, the last item is the return type. `add_ten` fits, so a type checker is happy; passing `len` would be flagged.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from collections.abc import Callable

type IntOp = Callable[[int], int]

def apply_twice(func: IntOp, value: int) -> int:
    return func(func(value))

print(apply_twice(add_ten, 5))
print(IntOp.__value__)
```

Callable hints get long fast. A `type` alias (Q09) names the shape once. Seniors import `Callable` from `collections.abc`; the `typing` version is an older alias of the same thing.

</details>

In [ ]:
def add_ten(n: int) -> int:
    return n + 10

# your code here


### NB26-Q07 · L2 · get_origin and get_args · Predict

`typing` can take a generic hint apart. Predict the output.

**What does this print?**

```python
from typing import get_origin, get_args

hint = dict[str, list[int]]
print(get_origin(hint))
print(get_args(hint))
print(get_args(get_args(hint)[1]))
```

<details><summary><b>Hint</b></summary>

The origin is the plain class before the brackets. The args are whatever is inside the brackets, as a tuple.

</details>

<details><summary><b>Expected output</b></summary>

```text
<class 'dict'>
(<class 'str'>, list[int])
(<class 'int'>,)
```

</details>

<details><summary><b>Solution</b></summary>

`get_origin()` gives the bare class (`dict`). `get_args()` gives the bracket contents as a tuple: `str` and `list[int]`. The second arg is itself a generic, so asking for *its* args gives `(int,)`. These two functions are how runtime validators walk through nested hints.

</details>

<details><summary><b>Senior dev solution</b></summary>

Use `get_origin()` and `get_args()` instead of poking at `__origin__` or `__args__`. They give consistent answers for unions, `Literal`, `Annotated` and the rest, across Python versions.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB26-Q08 · L2 · Literal · Write

`Mode` lists the only allowed file modes. Use `get_args()` to turn the
`Literal` into a tuple of allowed values and print it. Then print whether
`"r"` and `"x"` are allowed, on one line.
```text
('r', 'w', 'a')
True False
```

**Starting code (already in the cell below):**

```python
from typing import Literal, get_args

Mode = Literal["r", "w", "a"]
```

<details><summary><b>Hint</b></summary>

`get_args(Mode)` gives the tuple. Then the `in` operator answers "is it allowed?".

</details>

<details><summary><b>Expected output</b></summary>

```text
('r', 'w', 'a')
True False
```

</details>

<details><summary><b>Solution</b></summary>

```python
allowed = get_args(Mode)
print(allowed)
print("r" in allowed, "x" in allowed)
```

`Literal[...]` means "exactly one of these values", not just "any str". A type checker rejects `open_file("x")` at once. At runtime, `get_args()` recovers the values, so one definition drives both the static check and your runtime check.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
type Mode = Literal["r", "w", "a"]
allowed = get_args(Mode.__value__)
print(allowed)
print("r" in allowed, "x" in allowed)
```

Seniors keep one source of truth: the `Literal` feeds both mypy and the runtime check, so the two can never drift apart. With a `type` alias, ask for `.__value__` first. When the values need behaviour too, an `Enum` (NB21) is the next step up.

</details>

In [ ]:
from typing import Literal, get_args

Mode = Literal["r", "w", "a"]

# your code here


### NB26-Q09 · L2 · type aliases · Write

Create two aliases with the `type` statement: `Price` for `float`, and
`Basket` for a dict from `str` to `Price`. Print `Basket.__name__` and
`Basket.__value__` on one line, then use a `Basket` to print the total of `basket`.
```text
Basket dict[str, Price]
6.5
```

**Starting code (already in the cell below):**

```python
basket = {"bread": 2.0, "milk": 1.5, "eggs": 3.0}
```

<details><summary><b>Hint</b></summary>

The syntax is `type Name = some_type`. The alias has a `__name__` and a `__value__`. To total a dict's prices, sum its values.

</details>

<details><summary><b>Expected output</b></summary>

```text
Basket dict[str, Price]
6.5
```

</details>

<details><summary><b>Solution</b></summary>

```python
type Price = float
type Basket = dict[str, Price]

def basket_total(items: Basket) -> Price:
    return sum(items.values())

print(Basket.__name__, Basket.__value__)
print(basket_total(basket))
```

The `type` statement (3.12) makes a `TypeAliasType` object. Its value is evaluated lazily, only when you ask for `__value__`, so an alias can refer to names defined later. Notice `__value__` shows `Price`, not `float`: aliases keep their names, which makes error messages readable.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
type Price = float
type Basket = dict[str, Price]

def basket_total(items: Basket) -> Price:
    return round(sum(items.values()), 2)

print(Basket.__name__, Basket.__value__)
print(basket_total(basket))
```

An alias gives a domain name to a shape (`Basket` beats `dict[str, float]` in 20 signatures). Seniors also round money at the edge; for real money they'd use `Decimal` (NB16).

</details>

In [ ]:
basket = {"bread": 2.0, "milk": 1.5, "eggs": 3.0}

# your code here


### NB26-Q10 · L2 · Final · Predict

`MAX_RETRIES` is marked `Final`. Predict the output.

**What does this print?**

```python
from typing import Final

MAX_RETRIES: Final = 3
MAX_RETRIES = 5
print(MAX_RETRIES)
```

<details><summary><b>Hint</b></summary>

Who reads `Final`: Python, or a type checker?

</details>

<details><summary><b>Expected output</b></summary>

```text
5
```

</details>

<details><summary><b>Solution</b></summary>

It prints `5`. `Final` tells type checkers "never reassign this", and mypy reports the second line as an error. Python itself ignores it, like every other hint, so the reassignment happens.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from typing import Final

MAX_RETRIES: Final = 3
print(MAX_RETRIES)
```

`Final` plus an UPPER_CASE name is how seniors mark constants. The name warns humans, the hint lets mypy enforce it. For real runtime protection you would need a frozen dataclass or an `Enum` (NB21).

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

---

## L3 · Intermediate

### NB26-Q11 · L3 · TypeVar · Write

Write `first(items)` that returns the first item of a list. Make it
*generic* the classic way: create `T = TypeVar("T")`, annotate `items`
as `list[T]` and the return as `T`. Print `first([3, 1, 2])`,
`first(["tea", "milk"])`, then on a new line `get_type_hints(first)`.
```text
3 tea
{'items': list[~T], 'return': ~T}
```

**Starting code (already in the cell below):**

```python
from typing import TypeVar, get_type_hints
```

<details><summary><b>Hint</b></summary>

Create the `TypeVar` at module level, then use `T` in both hints. The same letter in two places means "the same type".

</details>

<details><summary><b>Expected output</b></summary>

```text
3 tea
{'items': list[~T], 'return': ~T}
```

</details>

<details><summary><b>Solution</b></summary>

```python
T = TypeVar("T")

def first(items: list[T]) -> T:
    return items[0]

print(first([3, 1, 2]), first(["tea", "milk"]))
print(get_type_hints(first))
```

A `TypeVar` is a placeholder. Using `T` for both the list items and the return value links them: a type checker knows `first([3, 1, 2])` is an `int` and `first(["tea"])` is a `str`. With `list[object] -> object` that link would be lost. The `~` in the repr marks a classic `TypeVar`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
T = TypeVar("T")

def first(items: list[T]) -> T:
    return items[0]

print(first([3, 1, 2]), first(["tea", "milk"]))
print(get_type_hints(first))
```

Same code, because this is the classic form you will meet in older codebases. Since 3.12, seniors write `def first[T](...)` instead (next question): no module-level `TypeVar`, and `T` is scoped to the one function. A bare `first([])` still raises `IndexError`: hints say nothing about empty lists.

</details>

In [ ]:
from typing import TypeVar, get_type_hints

# your code here


### NB26-Q12 · L3 · PEP 695 generics · Refactor

Rewrite this with the PEP 695 syntax (3.12): put the type parameter in
square brackets after the function name, and drop the `TypeVar` import
and the module-level `K` and `V`.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
from typing import TypeVar

K = TypeVar("K")
V = TypeVar("V")

def invert(mapping: dict[K, V]) -> dict[V, K]:
    result = {}
    for key, value in mapping.items():
        result[value] = key
    return result

print(invert({"red": 1, "green": 2}))
```

<details><summary><b>Hint</b></summary>

Two type parameters are written `def invert[K, V](...)`.

</details>

<details><summary><b>Expected output</b></summary>

```text
{1: 'red', 2: 'green'}
```

</details>

<details><summary><b>Solution</b></summary>

```python
def invert[K, V](mapping: dict[K, V]) -> dict[V, K]:
    result = {}
    for key, value in mapping.items():
        result[value] = key
    return result

print(invert({"red": 1, "green": 2}))
```

`def invert[K, V]` declares two type parameters that only exist inside this function. No import, no global names that another function might accidentally reuse. The behaviour is identical; only the hints changed, and Python stores the new parameters in `invert.__type_params__`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def invert[K, V](mapping: dict[K, V]) -> dict[V, K]:
    return {value: key for key, value in mapping.items()}

print(invert({"red": 1, "green": 2}))
```

With the noise gone, the body fits a dict comprehension (NB10). Seniors would also note that duplicate values silently overwrite: `invert` is only safe when values are unique.

</details>

In [ ]:
from typing import TypeVar

K = TypeVar("K")
V = TypeVar("V")

def invert(mapping: dict[K, V]) -> dict[V, K]:
    result = {}
    for key, value in mapping.items():
        result[value] = key
    return result

print(invert({"red": 1, "green": 2}))


### NB26-Q13 · L3 · Protocol · Write

Define a `@runtime_checkable` Protocol `HasArea` with one method
`area(self) -> float`. `Square` and `Circle` below do **not** inherit
from it. Print `isinstance()` of a `Square(2)` and of the string `"box"`
against `HasArea`, then the total area of `shapes` rounded to 2 places.
```text
True False
16.14
```

**Starting code (already in the cell below):**

```python
import math

class Square:
    def __init__(self, side: float) -> None:
        self.side = side
    def area(self) -> float:
        return self.side ** 2

class Circle:
    def __init__(self, radius: float) -> None:
        self.radius = radius
    def area(self) -> float:
        return math.pi * self.radius ** 2

shapes = [Square(2), Circle(1), Square(3)]
```

<details><summary><b>Hint</b></summary>

A Protocol is a class that inherits from `Protocol`; the method body is `...`. The decorator comes from `typing` too.

</details>

<details><summary><b>Expected output</b></summary>

```text
True False
16.14
```

</details>

<details><summary><b>Solution</b></summary>

```python
from typing import Protocol, runtime_checkable

@runtime_checkable
class HasArea(Protocol):
    def area(self) -> float: ...

def total_area(items: list[HasArea]) -> float:
    return sum(item.area() for item in items)

print(isinstance(Square(2), HasArea), isinstance("box", HasArea))
print(round(total_area(shapes), 2))
```

A Protocol describes a *shape*, not a family tree: anything with an `area()` method counts. This is duck typing that a type checker can verify ("structural typing"). `@runtime_checkable` lets `isinstance()` use it too; without the decorator, `isinstance()` against a Protocol raises `TypeError`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from typing import Protocol, runtime_checkable

@runtime_checkable
class HasArea(Protocol):
    def area(self) -> float: ...

def total_area(items: Iterable[HasArea]) -> float:
    return sum(item.area() for item in items)

from collections.abc import Iterable
print(isinstance(Square(2), HasArea), isinstance("box", HasArea))
print(round(total_area(shapes), 2))
```

Seniors prefer a Protocol over an ABC (NB21) when they don't own the classes: third-party types fit without inheriting anything. Taking `Iterable` instead of `list` lets callers pass tuples or generators. (The import sits below the function on purpose: in 3.14 hints are lazy, so the name only has to exist when something reads them.)

</details>

In [ ]:
import math

class Square:
    def __init__(self, side: float) -> None:
        self.side = side
    def area(self) -> float:
        return self.side ** 2

class Circle:
    def __init__(self, radius: float) -> None:
        self.radius = radius
    def area(self) -> float:
        return math.pi * self.radius ** 2

shapes = [Square(2), Circle(1), Square(3)]

# your code here


### NB26-Q14 · L3 · Protocol · Predict

`Label.area()` returns a string, not a float. Predict the output.

**What does this print?**

```python
from typing import Protocol, runtime_checkable

@runtime_checkable
class HasArea(Protocol):
    def area(self) -> float: ...

class Label:
    def area(self) -> str:
        return "big"

print(isinstance(Label(), HasArea))
```

<details><summary><b>Hint</b></summary>

What exactly can `isinstance()` check without calling the method?

</details>

<details><summary><b>Expected output</b></summary>

```text
True
```

</details>

<details><summary><b>Solution</b></summary>

It prints `True`. A runtime Protocol check only asks "does it have an attribute called `area`?". It never looks at parameters or return types, because that would mean reading hints, which Python doesn't enforce. mypy *would* reject `Label` where a `HasArea` is expected.

</details>

<details><summary><b>Senior dev solution</b></summary>

Treat `isinstance(x, SomeProtocol)` as a cheap "has these names" check. The real guarantee comes from the type checker. If the return type matters at runtime, check the returned value.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB26-Q15 · L3 · TypedDict · Write

Define a `TypedDict` called `Product` with keys `name: str` and
`price: float`. Build `item: Product` from the values below. Print
`type(item)`, then the sorted required keys (`Product.__required_keys__`),
then `item["name"]`.
```text
<class 'dict'>
['name', 'price']
Lamp
```

**Starting code (already in the cell below):**

```python
name = "Lamp"
price = 24.99
```

<details><summary><b>Hint</b></summary>

Write it like a class whose body only has annotated names. Create the value with a normal dict literal.

</details>

<details><summary><b>Expected output</b></summary>

```text
<class 'dict'>
['name', 'price']
Lamp
```

</details>

<details><summary><b>Solution</b></summary>

```python
from typing import TypedDict

class Product(TypedDict):
    name: str
    price: float

item: Product = {"name": name, "price": price}
print(type(item))
print(sorted(Product.__required_keys__))
print(item["name"])
```

A `TypedDict` gives a dict a fixed set of keys, each with its own type. At runtime the value is still a plain `dict`, so it's cheap and works with `json`. The class only exists for type checkers, plus a few runtime attributes like `__required_keys__` that validators can read (Q31).

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from typing import TypedDict

class Product(TypedDict):
    name: str
    price: float

item = Product(name=name, price=price)
print(type(item))
print(sorted(Product.__required_keys__))
print(item["name"])
```

Calling `Product(name=..., price=...)` builds the same plain dict but reads like a constructor. Seniors pick `TypedDict` for JSON-shaped data and a dataclass (NB21) when the data needs methods.

</details>

In [ ]:
name = "Lamp"
price = 24.99

# your code here


### NB26-Q16 · L3 · NewType · Error

`UserId` is a `NewType` based on `int`. Someone tries to check for it
with `isinstance()`. Which error does this raise?

**Which error does this raise?**

```python
from typing import NewType

UserId = NewType("UserId", int)
uid = UserId(42)
print(uid, type(uid))
print(isinstance(uid, UserId))
```

<details><summary><b>Hint</b></summary>

Look at what `type(uid)` prints. Is `UserId` a real class?

</details>

<details><summary><b>Expected output</b></summary>

```text
42 <class 'int'>
TypeError
```

</details>

<details><summary><b>Solution</b></summary>

The first print shows `42 <class 'int'>`: calling `UserId(42)` returns the plain int unchanged. `UserId` is not a class, so `isinstance()` raises `TypeError`. A `NewType` exists only for the type checker, which then refuses to mix up a `UserId` with an `OrderId` even though both are ints.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from typing import NewType

UserId = NewType("UserId", int)
uid = UserId(42)
print(uid, type(uid), UserId.__supertype__)
```

`NewType` costs nothing at runtime, which is why seniors use it for IDs. If you need a runtime check, check the supertype (`isinstance(uid, int)`), or make a real class when the difference must survive at runtime.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB26-Q17 · L3 · ClassVar · Write

In the dataclass `Order`, `TAX_RATE` should belong to the class, not be
a field. Annotate it with `ClassVar[float]`. Then print the names of the
dataclass fields and an order's total (`price * (1 + TAX_RATE)`).
```text
['item', 'price']
12.0
```

**Starting code (already in the cell below):**

```python
from dataclasses import dataclass, fields
from typing import ClassVar
```

<details><summary><b>Hint</b></summary>

`fields(Order)` gives field objects; each has a `.name`. Write the total as a method that reads `self.TAX_RATE`.

</details>

<details><summary><b>Expected output</b></summary>

```text
['item', 'price']
12.0
```

</details>

<details><summary><b>Solution</b></summary>

```python
@dataclass
class Order:
    TAX_RATE: ClassVar[float] = 0.2
    item: str
    price: float

    def total(self) -> float:
        return self.price * (1 + self.TAX_RATE)

print([f.name for f in fields(Order)])
print(Order("pen", 10.0).total())
```

Here a hint *does* change behaviour, because `@dataclass` reads the annotations. A `ClassVar` annotation tells it "skip this one", so `TAX_RATE` is not an `__init__` parameter and isn't in `fields()`. It also lets `TAX_RATE` have a default even though it comes before fields without one.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
@dataclass(frozen=True, slots=True)
class Order:
    TAX_RATE: ClassVar[float] = 0.2
    item: str
    price: float

    @property
    def total(self) -> float:
        return self.price * (1 + self.TAX_RATE)

print([f.name for f in fields(Order)])
print(Order("pen", 10.0).total)
```

A computed value reads best as a property (NB21), and a frozen, slotted dataclass prevents accidental changes. `ClassVar` is one of the few hints Python's own tools act on, together with `InitVar` and `KW_ONLY`.

</details>

In [ ]:
from dataclasses import dataclass, fields
from typing import ClassVar

# your code here


### NB26-Q18 · L3 · Annotations without values · Predict

`debug` is annotated but never given a value. Predict the output.

**What does this print?**

```python
class Config:
    debug: bool
    level: int = 1

print(hasattr(Config, "debug"), hasattr(Config, "level"))
print(Config.__annotations__)
```

<details><summary><b>Hint</b></summary>

Does an annotation on its own create anything you can look up?

</details>

<details><summary><b>Expected output</b></summary>

```text
False True
{'debug': <class 'bool'>, 'level': <class 'int'>}
```

</details>

<details><summary><b>Solution</b></summary>

It prints `False True`, then both hints. An annotation without `=` only records a hint; it does **not** create an attribute. That is why `Config.debug` would raise `AttributeError`. The hint itself is still stored, which is exactly what dataclasses and `TypedDict` rely on.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from typing import get_type_hints

class Config:
    debug: bool
    level: int = 1

print(hasattr(Config, "debug"), hasattr(Config, "level"))
print(get_type_hints(Config))
```

Seniors read class hints with `get_type_hints()`, which also merges hints from base classes. Reading `__annotations__` on a class directly only shows that one class.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB26-Q19 · L3 · What type checkers catch · Predict

The tests for `ratio` only ever used a non-zero `b`, and they all passed.
Predict the output.

**What does this print?**

```python
def ratio(a: int, b: int) -> float:
    if b == 0:
        return None
    return a / b

print(ratio(1, 2), ratio(1, 0))
```

<details><summary><b>Hint</b></summary>

Follow the `b == 0` branch. What does it return, and what does the hint promise?

</details>

<details><summary><b>Expected output</b></summary>

```text
0.5 None
```

</details>

<details><summary><b>Solution</b></summary>

It prints `0.5 None`. Python is happy, and tests that never pass `0` never see the problem. mypy reports it without running anything: `None` is not a `float`. Callers trusting the hint would later crash on `None * 2`. Type checkers check **every** path, tests only check the paths you remembered.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def ratio(a: int, b: int) -> float | None:
    if b == 0:
        return None
    return a / b

print(ratio(1, 2), ratio(1, 0))
```

Either make the hint honest (`float | None`), so the checker forces every caller to handle `None`, or raise `ZeroDivisionError` and keep `float`. Running mypy or pyright in CI is cheap insurance next to tests (NB28).

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB26-Q20 · L3 · Self · Write

Give `Cart` a method `add(self, item: str)` that appends the item and
returns `self`, annotated with `Self` from `typing`. Chain three calls,
print the items, then print the return hint via `get_type_hints(Cart.add)`.
```text
['tea', 'milk', 'bread']
typing.Self
```

**Starting code (already in the cell below):**

```python
from typing import Self, get_type_hints

class Cart:
    def __init__(self) -> None:
        self.items: list[str] = []
```

<details><summary><b>Hint</b></summary>

Add the method to the class in the starting code. A chained call looks like `Cart().add(...).add(...)`.

</details>

<details><summary><b>Expected output</b></summary>

```text
['tea', 'milk', 'bread']
typing.Self
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Cart:
    def __init__(self) -> None:
        self.items: list[str] = []

    def add(self, item: str) -> Self:
        self.items.append(item)
        return self

cart = Cart().add("tea").add("milk").add("bread")
print(cart.items)
print(get_type_hints(Cart.add)["return"])
```

`Self` means "an instance of whatever class this was called on". If `GiftCart` subclasses `Cart`, a type checker knows `GiftCart().add(...)` returns a `GiftCart`. Writing `-> Cart` would lose that.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Cart:
    def __init__(self) -> None:
        self.items: list[str] = []

    def add(self, *items: str) -> Self:
        self.items.extend(items)
        return self

cart = Cart().add("tea", "milk").add("bread")
print(cart.items)
print(get_type_hints(Cart.add)["return"])
```

`*items: str` annotates *each* extra argument, not the tuple. Seniors use `Self` for fluent builders and alternative constructors (`classmethod`s) so subclasses get the right type for free.

</details>

In [ ]:
from typing import Self, get_type_hints

class Cart:
    def __init__(self) -> None:
        self.items: list[str] = []

# your code here


---

## L4 · Somewhat Difficult

### NB26-Q21 · L4 · Generic classes · Write

Write a generic `Stack[T]` with the PEP 695 syntax (`class Stack[T]:`).
It keeps a `list[T]` and has `push(item: T) -> None`, `pop() -> T`,
`peek() -> T | None` (returns `None` when empty) and `__len__`.
Run the steps below and print as shown, then print `Stack.__type_params__`.
```text
3 c
c b 1
a None
(T,)
```
Steps: push `"a"`, `"b"`, `"c"`; print the length and `peek()`. Pop twice
and print both values and the new length. Pop once more and print it with
`peek()`.

<details><summary><b>Hint</b></summary>

Inside the class, `T` works like any other type name. `peek()` needs a check for an empty list before reading the last item.

</details>

<details><summary><b>Expected output</b></summary>

```text
3 c
c b 1
a None
(T,)
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Stack[T]:
    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)

    def pop(self) -> T:
        return self._items.pop()

    def peek(self) -> T | None:
        if self._items:
            return self._items[-1]
        return None

    def __len__(self) -> int:
        return len(self._items)

stack: Stack[str] = Stack()
stack.push("a")
stack.push("b")
stack.push("c")
print(len(stack), stack.peek())
first = stack.pop()
second = stack.pop()
print(first, second, len(stack))
print(stack.pop(), stack.peek())
print(Stack.__type_params__)
```

`class Stack[T]` makes `T` a type parameter for the whole class. `stack: Stack[str]` tells a type checker this stack holds strings, so `stack.push(3)` would be flagged and `stack.pop()` is known to be a `str`. At runtime nothing changes: the list happily holds anything. Python keeps the parameters in `__type_params__`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Stack[T]:
    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)

    def pop(self) -> T:
        if not self._items:
            raise IndexError("pop from empty Stack")
        return self._items.pop()

    def peek(self) -> T | None:
        return self._items[-1] if self._items else None

    def __len__(self) -> int:
        return len(self._items)

stack = Stack[str]()
for letter in "abc":
    stack.push(letter)
print(len(stack), stack.peek())
first, second = stack.pop(), stack.pop()
print(first, second, len(stack))
print(stack.pop(), stack.peek())
print(Stack.__type_params__)
```

`Stack[str]()` states the type at creation, so no separate annotation is needed. Raising a clear `IndexError` with the class name beats leaking the inner list's message. The `_items` underscore marks it private.

</details>

In [ ]:
# your code here


### NB26-Q22 · L4 · Constrained type parameters · Write

Write `clamp[N: (int, float)](value: N, low: N, high: N) -> N` that keeps
`value` between `low` and `high`. Print `clamp(15, 0, 10)` and
`clamp(-0.5, 0.0, 1.0)` on one line, then the constraints stored on the
type parameter (`clamp.__type_params__[0].__constraints__`).
```text
10 0.0
(<class 'int'>, <class 'float'>)
```

<details><summary><b>Hint</b></summary>

`min()` and `max()` together can clamp a value in one line.

</details>

<details><summary><b>Expected output</b></summary>

```text
10 0.0
(<class 'int'>, <class 'float'>)
```

</details>

<details><summary><b>Solution</b></summary>

```python
def clamp[N: (int, float)](value: N, low: N, high: N) -> N:
    return max(low, min(value, high))

print(clamp(15, 0, 10), clamp(-0.5, 0.0, 1.0))
print(clamp.__type_params__[0].__constraints__)
```

`N: (int, float)` is a *constraint*: `N` must be exactly `int` or exactly `float` for one call, so `clamp(15, 0, 10)` returns an `int` and a type checker rejects `clamp("a", "b", "c")`. Writing `N: SomeClass` (no tuple) is a *bound* instead: any subclass of `SomeClass` is allowed, and it is stored in `__bound__`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def clamp[N: (int, float)](value: N, low: N, high: N) -> N:
    if low > high:
        raise ValueError("low must not exceed high")
    return max(low, min(value, high))

print(clamp(15, 0, 10), clamp(-0.5, 0.0, 1.0))
print(clamp.__type_params__[0].__constraints__)
```

Hints check types, not values: nothing stops `clamp(5, 10, 0)`. Seniors guard value rules with a real check. They use constraints rarely; a bound to a Protocol is the more common tool.

</details>

In [ ]:
# your code here


### NB26-Q23 · L4 · overload · Write

`parse` turns a string into an `int`, and a list of strings into a list
of ints. Declare both signatures with `@overload` (bodies are `...`),
then write the real implementation underneath. Print `parse("42")`,
`parse(["1", "2"])`, then how many overloads `get_overloads(parse)` finds.
```text
42 [1, 2]
2
```

**Starting code (already in the cell below):**

```python
from typing import overload, get_overloads
```

<details><summary><b>Hint</b></summary>

Write the two `@overload` stubs first, then a normal `def parse(raw)` with no decorator that uses `isinstance()` to pick a branch.

</details>

<details><summary><b>Expected output</b></summary>

```text
42 [1, 2]
2
```

</details>

<details><summary><b>Solution</b></summary>

```python
@overload
def parse(raw: str) -> int: ...
@overload
def parse(raw: list[str]) -> list[int]: ...

def parse(raw):
    if isinstance(raw, str):
        return int(raw)
    return [int(part) for part in raw]

print(parse("42"), parse(["1", "2"]))
print(len(get_overloads(parse)))
```

With one signature `str | list[str] -> int | list[int]`, a checker could not tell which result you get. The `@overload` stubs pair each input type with its output, so `parse("42")` is known to be an `int`. Only the last, undecorated `def` runs; the stubs are recorded and `get_overloads()` can list them.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
@overload
def parse(raw: str) -> int: ...
@overload
def parse(raw: list[str]) -> list[int]: ...

def parse(raw: str | list[str]) -> int | list[int]:
    if isinstance(raw, str):
        return int(raw)
    return [int(part) for part in raw]

print(parse("42"), parse(["1", "2"]))
print(len(get_overloads(parse)))
```

Seniors also annotate the implementation with the union, so the body itself is checked. For behaviour that differs by type at *runtime*, `functools.singledispatch` (NB23) is the dispatching tool; `@overload` only describes types.

</details>

In [ ]:
from typing import overload, get_overloads

# your code here


### NB26-Q24 · L4 · TypeIs · Write [3.13+]

Write `is_str(value: object) -> TypeIs[str]` that returns whether
`value` is a string. Use it to keep only the strings in `entries`,
upper-cased. Print the result, then the return hint of `is_str`.
```text
['TEA', 'MILK']
typing.TypeIs[str]
```

**Starting code (already in the cell below):**

```python
from typing import TypeIs, get_type_hints

entries: list[object] = ["tea", 3, "milk", None]
```

<details><summary><b>Hint</b></summary>

`is_str` is one `isinstance()` call. A list comprehension with an `if` does the filtering.

</details>

<details><summary><b>Expected output</b></summary>

```text
['TEA', 'MILK']
typing.TypeIs[str]
```

</details>

<details><summary><b>Solution</b></summary>

```python
def is_str(value: object) -> TypeIs[str]:
    return isinstance(value, str)

print([entry.upper() for entry in entries if is_str(entry)])
print(get_type_hints(is_str)["return"])
```

At runtime `is_str` returns a plain `bool`. The `TypeIs[str]` hint tells a type checker more: after `if is_str(entry)`, `entry` is a `str`, so `.upper()` is allowed. With `-> bool` the checker would still think `entry` is an `object` and reject `.upper()`. `TypeIs` (3.13) narrows the `else` branch too, unlike the older `TypeGuard`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def is_str(value: object) -> TypeIs[str]:
    return isinstance(value, str)

shouted = [entry.upper() for entry in entries if is_str(entry)]
print(shouted)
print(get_type_hints(is_str)["return"])
```

For a single `isinstance()` you don't need a helper: checkers narrow `isinstance()` directly. Seniors write `TypeIs` functions for checks too complex to inline, like "a dict with the right keys".

</details>

In [ ]:
from typing import TypeIs, get_type_hints

entries: list[object] = ["tea", 3, "milk", None]

# your code here


### NB26-Q25 · L4 · override · Write

`EmailNotifier` replaces `send()` from `Notifier`. Mark the method with
`@override` from `typing`. Print the result of sending `"Order shipped"`,
then the `__override__` flag the decorator sets on the method.
```text
email: Order shipped
True
```

**Starting code (already in the cell below):**

```python
from typing import override

class Notifier:
    def send(self, message: str) -> str:
        return f"log: {message}"
```

<details><summary><b>Hint</b></summary>

The decorator goes directly above the `def` in the subclass. Read the flag from `EmailNotifier.send`.

</details>

<details><summary><b>Expected output</b></summary>

```text
email: Order shipped
True
```

</details>

<details><summary><b>Solution</b></summary>

```python
class EmailNotifier(Notifier):
    @override
    def send(self, message: str) -> str:
        return f"email: {message}"

print(EmailNotifier().send("Order shipped"))
print(EmailNotifier.send.__override__)
```

`@override` (3.12) says "this method replaces one in a base class". If someone later renames `Notifier.send` to `notify`, a type checker reports that `EmailNotifier.send` no longer overrides anything, instead of the bug silently turning into a new, unused method. At runtime it only sets `__override__ = True`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class EmailNotifier(Notifier):
    @override
    def send(self, message: str) -> str:
        return f"email: {message}"

notifier: Notifier = EmailNotifier()
print(notifier.send("Order shipped"))
print(EmailNotifier.send.__override__)
```

Annotating the variable as the base type (`notifier: Notifier`) is how real code uses subclasses: callers depend on the base, the override supplies the behaviour. Many teams turn on a checker rule that makes `@override` mandatory.

</details>

In [ ]:
from typing import override

class Notifier:
    def send(self, message: str) -> str:
        return f"log: {message}"

# your code here


### NB26-Q26 · L4 · Deferred annotations · Error [3.14+]

`Parcel` is never defined anywhere. In Python 3.14, which error does this
raise, and what is printed first?

**Which error does this raise?**

```python
def ship(order: Parcel) -> None:
    print("shipping", order)

ship("box-1")
print(ship.__annotations__)
```

<details><summary><b>Hint</b></summary>

In 3.14, when are annotations evaluated: when the `def` runs, or when someone reads them?

</details>

<details><summary><b>Expected output</b></summary>

```text
shipping box-1
NameError
```

</details>

<details><summary><b>Solution</b></summary>

It prints `shipping box-1`, then raises `NameError`. Since 3.14, annotations are **deferred**: Python saves how to compute them and only does it when you read `__annotations__` (or call `get_type_hints()`). So the `def` and the call work, and the missing name only bites on the read. In 3.13 and earlier the `def` line itself raised `NameError`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from annotationlib import get_annotations, Format

def ship(order: Parcel) -> None:
    print("shipping", order)

ship("box-1")
print(get_annotations(ship, format=Format.STRING))
```

`annotationlib` (3.14) can read hints without evaluating them. `Format.STRING` gives the source text, which never fails. Seniors use it in documentation tools; for validation they want real objects and fix the missing name instead.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB26-Q27 · L4 · Forward references · Write [3.14+]

Write a dataclass `Node` with `value: int` and `next: Node | None = None`.
It refers to itself, with no quotes. Build the chain 1 -> 2 -> 3, walk it
and print the values on one line, then print its annotations as text with
`get_annotations(Node, format=Format.STRING)`.
```text
1 2 3
{'value': 'int', 'next': 'Node | None'}
```

**Starting code (already in the cell below):**

```python
from dataclasses import dataclass
from annotationlib import get_annotations, Format
```

<details><summary><b>Hint</b></summary>

Build from the end: `Node(1, Node(2, Node(3)))`. Walk with a `while` loop that follows `.next` until it is `None`, collecting values in a list.

</details>

<details><summary><b>Expected output</b></summary>

```text
1 2 3
{'value': 'int', 'next': 'Node | None'}
```

</details>

<details><summary><b>Solution</b></summary>

```python
@dataclass
class Node:
    value: int
    next: Node | None = None

node = Node(1, Node(2, Node(3)))
values = []
while node is not None:
    values.append(node.value)
    node = node.next
print(*values)
print(get_annotations(Node, format=Format.STRING))
```

While the class body runs, the name `Node` doesn't exist yet. Before 3.14 you had to write `next: "Node | None"` in quotes (a *forward reference*), or the class raised `NameError`. With deferred annotations the hint is only evaluated later, when `Node` exists, so the quotes are no longer needed.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
@dataclass(slots=True)
class Node:
    value: int
    next: Node | None = None

def walk(node: Node | None):
    while node is not None:
        yield node.value
        node = node.next

print(*walk(Node(1, Node(2, Node(3)))))
print(get_annotations(Node, format=Format.STRING))
```

A generator (NB22) separates *walking* from *using* the values. Code that must also run on 3.13 or older keeps the quotes or adds `from __future__ import annotations` at the top of the file.

</details>

In [ ]:
from dataclasses import dataclass
from annotationlib import get_annotations, Format

# your code here


### NB26-Q28 · L4 · Runtime checks with get_origin · Fix

`check(value, hint)` should say whether a value matches a hint, including
list hints like `list[int]`, where every item must match. It crashes.
Fix it so it prints:
```text
True
True
False
```

**Buggy code (copied into the cell below):**

```python
def check(value, hint):
    return isinstance(value, hint)

print(check(3, int))
print(check([1, 2], list[int]))
print(check([1, "2"], list[int]))
```

<details><summary><b>Hint</b></summary>

`get_origin(list[int])` is `list` and `get_origin(int)` is `None`. Check the container with the origin, then each item against `get_args(hint)[0]`.

</details>

<details><summary><b>Expected output</b></summary>

```text
True
True
False
```

</details>

<details><summary><b>Solution</b></summary>

```python
from typing import get_origin, get_args

def check(value, hint):
    origin = get_origin(hint)
    if origin is None:
        return isinstance(value, hint)
    if not isinstance(value, origin):
        return False
    item_hint = get_args(hint)[0]
    return all(check(item, item_hint) for item in value)

print(check(3, int))
print(check([1, 2], list[int]))
print(check([1, "2"], list[int]))
```

`isinstance()` can't take `list[int]` (Q05). The fix splits the hint: `get_origin()` gives the plain `list` for the container check, and `get_args()` gives `int` for the items. Calling `check` again for each item (recursion, NB11) means `list[list[int]]` works too.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from typing import get_origin, get_args

def check(value: object, hint: type) -> bool:
    origin = get_origin(hint)
    if origin is None:
        return isinstance(value, hint)
    if not isinstance(value, origin):
        return False
    (item_hint,) = get_args(hint)
    return all(check(item, item_hint) for item in value)

print(check(3, int))
print(check([1, 2], list[int]))
print(check([1, "2"], list[int]))
```

`(item_hint,) = get_args(hint)` unpacks *exactly one* argument, so `dict[str, int]` fails loudly instead of being checked wrongly. A real validator handles each origin (`dict`, `tuple`, unions) on its own branch.

</details>

In [ ]:
def check(value, hint):
    return isinstance(value, hint)

print(check(3, int))
print(check([1, 2], list[int]))
print(check([1, "2"], list[int]))


### NB26-Q29 · L4 · ClassVar in dataclasses · Fix

`count` is meant to be a shared counter of how many tickets exist, not
a field. Instead, the title ends up in the wrong place. Fix it with one
hint change so it prints:
```text
Ticket(title='Login broken')
Ticket(title='Slow page')
tickets: 2
```

**Buggy code (copied into the cell below):**

```python
from dataclasses import dataclass

@dataclass
class Ticket:
    count: int = 0
    title: str = "untitled"

    def __post_init__(self) -> None:
        Ticket.count += 1

print(Ticket("Login broken"))
print(Ticket("Slow page"))
print("tickets:", Ticket.count)
```

<details><summary><b>Hint</b></summary>

`@dataclass` turns every annotated name into a field. Which hint (Q17) tells it to leave one alone?

</details>

<details><summary><b>Expected output</b></summary>

```text
Ticket(title='Login broken')
Ticket(title='Slow page')
tickets: 2
```

</details>

<details><summary><b>Solution</b></summary>

```python
from dataclasses import dataclass
from typing import ClassVar

@dataclass
class Ticket:
    count: ClassVar[int] = 0
    title: str = "untitled"

    def __post_init__(self) -> None:
        Ticket.count += 1

print(Ticket("Login broken"))
print(Ticket("Slow page"))
print("tickets:", Ticket.count)
```

With `count: int`, `count` became the first `__init__` parameter, so `"Login broken"` was stored as the count and the title stayed `"untitled"`. Changing the hint to `ClassVar[int]` removes it from the fields. A hint changed the program's behaviour, because `@dataclass` reads hints when it builds the class.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from dataclasses import dataclass
from typing import ClassVar

@dataclass(kw_only=True)
class Ticket:
    count: ClassVar[int] = 0
    title: str

    def __post_init__(self) -> None:
        type(self).count += 1

print(Ticket(title="Login broken"))
print(Ticket(title="Slow page"))
print("tickets:", Ticket.count)
```

`kw_only=True` forces `Ticket(title=...)`, which makes this whole class of "wrong slot" bug impossible. A title with no default is also more honest than `"untitled"`.

</details>

In [ ]:
from dataclasses import dataclass

@dataclass
class Ticket:
    count: int = 0
    title: str = "untitled"

    def __post_init__(self) -> None:
        Ticket.count += 1

print(Ticket("Login broken"))
print(Ticket("Slow page"))
print("tickets:", Ticket.count)


### NB26-Q30 · L4 · Type-parameter defaults · Write [3.13+]

Write `class Response[T = str]` that stores a `body: T` in `__init__`.
Read its type parameter and print its name, whether it has a default
and the default. Then print the body of `Response("ok")`.
```text
T True <class 'str'>
ok
```

<details><summary><b>Hint</b></summary>

The type parameter is `Response.__type_params__[0]`. It has `__name__`, `has_default()` and `__default__`.

</details>

<details><summary><b>Expected output</b></summary>

```text
T True <class 'str'>
ok
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Response[T = str]:
    def __init__(self, body: T) -> None:
        self.body = body

param = Response.__type_params__[0]
print(param.__name__, param.has_default(), param.__default__)
print(Response("ok").body)
```

A type-parameter default (3.13) says which type to assume when nobody says. A plain `Response` hint now means `Response[str]` to a checker, instead of `Response[Any]`, which would switch off checking for the body.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Response[T = str]:
    def __init__(self, body: T, status: int = 200) -> None:
        self.body = body
        self.status = status

param = Response.__type_params__[0]
print(param.__name__, param.has_default(), param.__default__)
print(Response("ok").body)
```

Defaults let a library make a class generic later without breaking every user who wrote plain `Response`. Seniors pick the default that the most common use needs.

</details>

In [ ]:
# your code here


---

## L5 · Hard

### NB26-Q31 · L5 · TypedDict validator · Write

Data from a file claims to be `Product` records. Write
`validate(data, schema) -> list[str]` that reads the schema's hints with
`get_type_hints()` and returns a list of problems, in this order:

1. each missing **required** key (sorted): `missing name`
2. each key not in the schema (sorted): `unexpected colour`
3. each present key with the wrong type, in schema order:
   `price: expected float, got str`

`bool` is a subclass of `int`, but `True` must **not** pass as a stock
count. Print one line per record:
```text
record 1: ok
record 2: price: expected float, got str; stock: expected int, got bool
record 3: missing name; unexpected colour
```

**Starting code (already in the cell below):**

```python
from typing import TypedDict, NotRequired, get_type_hints

class Product(TypedDict):
    name: str
    price: float
    stock: NotRequired[int]

records = [
    {"name": "Lamp", "price": 24.99},
    {"name": "Desk", "price": "120", "stock": True},
    {"price": 5.0, "stock": 3, "colour": "red"},
]
```

<details><summary><b>Hint</b></summary>

`schema.__required_keys__` holds the keys that must be there; `get_type_hints()` strips `NotRequired` and gives the plain types. For the type check, treat a `bool` value as wrong unless the hint is `bool` itself.

</details>

<details><summary><b>Expected output</b></summary>

```text
record 1: ok
record 2: price: expected float, got str; stock: expected int, got bool
record 3: missing name; unexpected colour
```

</details>

<details><summary><b>Solution</b></summary>

```python
def validate(data, schema):
    hints = get_type_hints(schema)
    problems = []
    for key in sorted(schema.__required_keys__):
        if key not in data:
            problems.append(f"missing {key}")
    for key in sorted(data):
        if key not in hints:
            problems.append(f"unexpected {key}")
    for key, hint in hints.items():
        if key not in data:
            continue
        value = data[key]
        wrong_bool = isinstance(value, bool) and hint is not bool
        if wrong_bool or not isinstance(value, hint):
            got = type(value).__name__
            problems.append(f"{key}: expected {hint.__name__}, got {got}")
    return problems

for number, record in enumerate(records, start=1):
    problems = validate(record, Product)
    if problems:
        print(f"record {number}: " + "; ".join(problems))
    else:
        print(f"record {number}: ok")
```

A `TypedDict` is a plain dict at runtime, so nothing stopped these records being built. The class still carries everything a validator needs: `__required_keys__` (where `NotRequired` keys are left out) and the hints. The trap is `isinstance(True, int)`, which is `True` because `bool` subclasses `int`; without the extra check, `stock: True` would pass.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def validate(data: dict[str, object], schema: type) -> list[str]:
    hints = get_type_hints(schema)
    problems = [f"missing {key}" for key in sorted(schema.__required_keys__ - data.keys())]
    problems += [f"unexpected {key}" for key in sorted(data.keys() - hints.keys())]
    for key, hint in hints.items():
        if key not in data:
            continue
        value = data[key]
        if (isinstance(value, bool) and hint is not bool) or not isinstance(value, hint):
            problems.append(f"{key}: expected {hint.__name__}, got {type(value).__name__}")
    return problems

for number, record in enumerate(records, start=1):
    print(f"record {number}: {'; '.join(validate(record, Product)) or 'ok'}")
```

Set differences on `keys()` views (NB06) find missing and extra keys in one expression each. `'; '.join(...) or 'ok'` works because an empty string is falsy. In production, pydantic or msgspec build this validator for you from the same hints.

</details>

In [ ]:
from typing import TypedDict, NotRequired, get_type_hints

class Product(TypedDict):
    name: str
    price: float
    stock: NotRequired[int]

records = [
    {"name": "Lamp", "price": 24.99},
    {"name": "Desk", "price": "120", "stock": True},
    {"price": 5.0, "stock": 3, "colour": "red"},
]

# your code here


### NB26-Q32 · L5 · Enforcing hints with a decorator · Write

Write a decorator `@enforce` that checks every argument against the
function's hints before calling it, and raises
`TypeError("qty must be int, got bool")` (that format) on a mismatch.
Use `inspect.signature(func).bind(*args, **kwargs)` to match arguments to
names, and `get_type_hints()` for the types. Hints like `str | None` must
work, and `True` must not pass as an `int`. Run the calls below and print:
```text
pen +5
rejected: qty must be int, got bool
rejected: qty must be int, got str
pen +2 (urgent)
```

**Starting code (already in the cell below):**

```python
import functools
import inspect
from typing import get_type_hints

calls = [("pen", 5), ("pen", True), ("pen", "5"), ("pen", 2, "urgent")]
```

<details><summary><b>Hint</b></summary>

`bound.arguments` is a dict of name to value. Skip the `'return'` hint. `isinstance(value, str | None)` already handles unions; only the `bool` case needs an extra test.

</details>

<details><summary><b>Expected output</b></summary>

```text
pen +5
rejected: qty must be int, got bool
rejected: qty must be int, got str
pen +2 (urgent)
```

</details>

<details><summary><b>Solution</b></summary>

```python
def enforce(func):
    sig = inspect.signature(func)
    hints = get_type_hints(func)

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        bound = sig.bind(*args, **kwargs)
        for name, value in bound.arguments.items():
            hint = hints.get(name)
            if hint is None:
                continue
            wrong_bool = isinstance(value, bool) and hint is int
            if wrong_bool or not isinstance(value, hint):
                got = type(value).__name__
                raise TypeError(f"{name} must be {hint.__name__}, got {got}")
        return func(*args, **kwargs)
    return wrapper

@enforce
def restock(item: str, qty: int, note: str | None = None) -> str:
    text = f"{item} +{qty}"
    if note is not None:
        text += f" ({note})"
    return text

for args in calls:
    try:
        print(restock(*args))
    except TypeError as error:
        print("rejected:", error)
```

Hints are never enforced (Q02), but they are data, so a decorator can enforce them. `signature().bind()` maps positional and keyword arguments to parameter names exactly as Python would. `isinstance()` accepts a union like `str | None` directly. The hints are read once, when decorating, not on every call. Careful: `hint.__name__` would fail for a union in the message; this example only ever fails on plain classes.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def enforce(func):
    sig = inspect.signature(func)
    hints = {k: v for k, v in get_type_hints(func).items() if k != "return"}

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        for name, value in sig.bind(*args, **kwargs).arguments.items():
            hint = hints.get(name)
            if hint is None:
                continue
            if (isinstance(value, bool) and hint is int) or not isinstance(value, hint):
                label = getattr(hint, "__name__", str(hint))
                raise TypeError(f"{name} must be {label}, got {type(value).__name__}")
        return func(*args, **kwargs)
    return wrapper

@enforce
def restock(item: str, qty: int, note: str | None = None) -> str:
    return f"{item} +{qty}" + (f" ({note})" if note is not None else "")

for args in calls:
    try:
        print(restock(*args))
    except TypeError as error:
        print("rejected:", error)
```

`getattr(hint, "__name__", str(hint))` keeps the error message working for unions such as `str | None`, which have no `__name__`. Seniors only enforce hints at trust boundaries (API handlers, config loading): checking every internal call costs time and the type checker already covers it.

</details>

In [ ]:
import functools
import inspect
from typing import get_type_hints

calls = [("pen", 5), ("pen", True), ("pen", "5"), ("pen", 2, "urgent")]

# your code here


### NB26-Q33 · L5 · ReadOnly · Write [3.13+]

In `Account`, `id` is `ReadOnly`. Write `apply_changes(record, changes,
schema)` that copies allowed changes into a **new** dict and returns it
with a sorted list of blocked keys (any key in `schema.__readonly_keys__`).
The original `record` must not change. Print:
```text
updated: {'id': 7, 'owner': 'Grace', 'balance': 50.0}
blocked: ['id']
original: {'id': 7, 'owner': 'Ada', 'balance': 10.0}
```

**Starting code (already in the cell below):**

```python
from typing import TypedDict, ReadOnly

class Account(TypedDict):
    id: ReadOnly[int]
    owner: str
    balance: float

record: Account = {"id": 7, "owner": "Ada", "balance": 10.0}
changes = {"balance": 50.0, "id": 99, "owner": "Grace"}
```

<details><summary><b>Hint</b></summary>

Start from `dict(record)` so you never touch the original. Loop over `changes` and decide per key.

</details>

<details><summary><b>Expected output</b></summary>

```text
updated: {'id': 7, 'owner': 'Grace', 'balance': 50.0}
blocked: ['id']
original: {'id': 7, 'owner': 'Ada', 'balance': 10.0}
```

</details>

<details><summary><b>Solution</b></summary>

```python
def apply_changes(record, changes, schema):
    updated = dict(record)
    blocked = []
    for key, value in changes.items():
        if key in schema.__readonly_keys__:
            blocked.append(key)
        else:
            updated[key] = value
    return updated, sorted(blocked)

updated, blocked = apply_changes(record, changes, Account)
print("updated:", updated)
print("blocked:", blocked)
print("original:", record)
```

`ReadOnly` (3.13) tells a type checker that code must not assign to `record["id"]`. Python itself would allow it, so the runtime rule comes from `__readonly_keys__`, a frozenset the `TypedDict` class builds from the hints. Copying first is the trap: `updated = record` would just be a second name for the same dict, and the original would change too.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def apply_changes(record: Account, changes: dict[str, object],
                  schema: type) -> tuple[dict[str, object], list[str]]:
    blocked = sorted(changes.keys() & schema.__readonly_keys__)
    allowed = {k: v for k, v in changes.items() if k not in schema.__readonly_keys__}
    return record | allowed, blocked

updated, blocked = apply_changes(record, changes, Account)
print("updated:", updated)
print("blocked:", blocked)
print("original:", record)
```

`record | allowed` (NB06) builds a new merged dict and keeps the key order of `record`, so the original is safe by construction. A set intersection finds the blocked keys in one step.

</details>

In [ ]:
from typing import TypedDict, ReadOnly

class Account(TypedDict):
    id: ReadOnly[int]
    owner: str
    balance: float

record: Account = {"id": 7, "owner": "Ada", "balance": 10.0}
changes = {"balance": 50.0, "id": 99, "owner": "Grace"}

# your code here


### NB26-Q34 · L5 · Generic type aliases · Refactor

The same long hint appears five times. Refactor with two generic `type`
aliases: `Row[T]` for `tuple[str, T]` and `Table[T]` for `list[Row[T]]`,
and make both functions generic with `[T]`. Keep the output the same.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
def best(rows: list[tuple[str, float]]) -> tuple[str, float]:
    top = rows[0]
    for row in rows:
        if row[1] > top[1]:
            top = row
    return top

def labels(rows: list[tuple[str, float]]) -> list[str]:
    return [row[0] for row in rows]

scores: list[tuple[str, float]] = [("Ada", 91.5), ("Linus", 88.0), ("Grace", 91.5)]
print(best(scores))
print(labels(scores))
print(best([("x", 3), ("y", 7)]))
```

<details><summary><b>Hint</b></summary>

A generic alias takes its own type parameter: `type Row[T] = tuple[str, T]`. Then `Table[float]` reads as "a table of floats".

</details>

<details><summary><b>Expected output</b></summary>

```text
('Ada', 91.5)
['Ada', 'Linus', 'Grace']
('y', 7)
```

</details>

<details><summary><b>Solution</b></summary>

```python
type Row[T] = tuple[str, T]
type Table[T] = list[Row[T]]

def best[T](rows: Table[T]) -> Row[T]:
    top = rows[0]
    for row in rows:
        if row[1] > top[1]:
            top = row
    return top

def labels[T](rows: Table[T]) -> list[str]:
    return [row[0] for row in rows]

scores: Table[float] = [("Ada", 91.5), ("Linus", 88.0), ("Grace", 91.5)]
print(best(scores))
print(labels(scores))
print(best([("x", 3), ("y", 7)]))
```

The original hints also lied: `best` was called with ints, which `list[tuple[str, float]]` happens to accept, but the return type then claimed `float`. With `[T]`, `best` returns a `Row[int]` for int rows. The trap to keep: `>` (not `>=`) means a tie keeps the **first** best row, so `Ada` wins over `Grace`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from operator import itemgetter

type Row[T] = tuple[str, T]
type Table[T] = list[Row[T]]

def best[T](rows: Table[T]) -> Row[T]:
    return max(rows, key=itemgetter(1))

def labels[T](rows: Table[T]) -> list[str]:
    return [label for label, _ in rows]

scores: Table[float] = [("Ada", 91.5), ("Linus", 88.0), ("Grace", 91.5)]
print(best(scores))
print(labels(scores))
print(best([("x", 3), ("y", 7)]))
```

`max()` also returns the first of equal items, so the tie rule survives. Strictly, `T` should be bound to something comparable (a Protocol with `__lt__`); seniors add that bound in library code so `best` on un-orderable values is caught by the checker.

</details>

In [ ]:
def best(rows: list[tuple[str, float]]) -> tuple[str, float]:
    top = rows[0]
    for row in rows:
        if row[1] > top[1]:
            top = row
    return top

def labels(rows: list[tuple[str, float]]) -> list[str]:
    return [row[0] for row in rows]

scores: list[tuple[str, float]] = [("Ada", 91.5), ("Linus", 88.0), ("Grace", 91.5)]
print(best(scores))
print(labels(scores))
print(best([("x", 3), ("y", 7)]))


---

## Mini-project

Pull together everything from this notebook.

## NB26-P · L5 · Mini-project: Typed Repository Layer · Write

Build a small, fully typed data layer for a library: a generic
`Repository[T]` that can sit on top of different storage backends.

1. `HasId`: a `@runtime_checkable` Protocol with one attribute, `id: int`.
2. `Storage[T]`: a `@runtime_checkable` generic Protocol
   (`class Storage[T](Protocol):`) with `save(self, key: int, item: T) -> None`,
   `load(self, key: int) -> T | None` and `keys(self) -> list[int]`.
3. `MemoryStorage[T]`: keeps items in a `dict[int, T]`. `keys()` returns them sorted.
4. `AuditedStorage[T]`: wraps another `Storage[T]`, passes every call on,
   and appends `"save 3"` / `"load 3"` style lines to a `log: list[str]`.
5. `Repository[T: HasId]`: takes a `Storage[T]`. `add(item)` raises
   `TypeError("str has no id")` (using the type's name) when the item isn't
   a `HasId`, else saves it under `item.id`. `get(item_id)` returns the item or
   `None`. `all()` returns every item in id order.

Then, using `Book` and `Member` from the starting code:

- a `Repository[Book]` on a `MemoryStorage`: add `Book(2, "Emma")` and
  `Book(1, "Dune")`, try to add the string `"Oops"`, then print as below;
- a `Repository[Member]` on an `AuditedStorage` around a `MemoryStorage`:
  add `Member(5, "Linus")` and `Member(3, "Ada")`, get member 3, then print
  the member names (in id order) and the audit log;
- finally print whether a `MemoryStorage()` counts as a `Storage`, and the
  name of `Repository`'s type-parameter bound.

```text
rejected: str has no id
books: [Book(id=1, title='Dune'), Book(id=2, title='Emma')]
get 2: Book(id=2, title='Emma')
get 9: None
members: ['Ada', 'Linus']
audit: ['save 5', 'save 3', 'load 3', 'load 3', 'load 5']
storage ok: True
bound: HasId
```

**Starting code (already in the cell below):**

```python
from dataclasses import dataclass
from typing import Protocol, runtime_checkable

@dataclass
class Book:
    id: int
    title: str

@dataclass
class Member:
    id: int
    name: str
```

<details><summary><b>Hint</b></summary>

Write `all()` on top of `keys()` and `load()`, so it works with any backend. That is also why the audit log shows the `load` lines after `get 3`. The bound lives in `Repository.__type_params__[0].__bound__`.

</details>

<details><summary><b>Expected output</b></summary>

```text
rejected: str has no id
books: [Book(id=1, title='Dune'), Book(id=2, title='Emma')]
get 2: Book(id=2, title='Emma')
get 9: None
members: ['Ada', 'Linus']
audit: ['save 5', 'save 3', 'load 3', 'load 3', 'load 5']
storage ok: True
bound: HasId
```

</details>

<details><summary><b>Solution</b></summary>

```python
@runtime_checkable
class HasId(Protocol):
    id: int

@runtime_checkable
class Storage[T](Protocol):
    def save(self, key: int, item: T) -> None: ...
    def load(self, key: int) -> T | None: ...
    def keys(self) -> list[int]: ...

class MemoryStorage[T]:
    def __init__(self) -> None:
        self._data: dict[int, T] = {}

    def save(self, key: int, item: T) -> None:
        self._data[key] = item

    def load(self, key: int) -> T | None:
        return self._data.get(key)

    def keys(self) -> list[int]:
        return sorted(self._data)

class AuditedStorage[T]:
    def __init__(self, inner: Storage[T]) -> None:
        self.inner = inner
        self.log: list[str] = []

    def save(self, key: int, item: T) -> None:
        self.log.append(f"save {key}")
        self.inner.save(key, item)

    def load(self, key: int) -> T | None:
        self.log.append(f"load {key}")
        return self.inner.load(key)

    def keys(self) -> list[int]:
        return self.inner.keys()

class Repository[T: HasId]:
    def __init__(self, storage: Storage[T]) -> None:
        self.storage = storage

    def add(self, item: T) -> None:
        if not isinstance(item, HasId):
            raise TypeError(f"{type(item).__name__} has no id")
        self.storage.save(item.id, item)

    def get(self, item_id: int) -> T | None:
        return self.storage.load(item_id)

    def all(self) -> list[T]:
        items = []
        for key in self.storage.keys():
            item = self.storage.load(key)
            if item is not None:
                items.append(item)
        return items

books: Repository[Book] = Repository(MemoryStorage())
books.add(Book(2, "Emma"))
books.add(Book(1, "Dune"))
try:
    books.add("Oops")
except TypeError as error:
    print("rejected:", error)
print("books:", books.all())
print("get 2:", books.get(2))
print("get 9:", books.get(9))

audited: AuditedStorage[Member] = AuditedStorage(MemoryStorage())
members: Repository[Member] = Repository(audited)
members.add(Member(5, "Linus"))
members.add(Member(3, "Ada"))
members.get(3)
names = []
for member in members.all():
    names.append(member.name)
print("members:", names)
print("audit:", audited.log)

print("storage ok:", isinstance(MemoryStorage(), Storage))
print("bound:", Repository.__type_params__[0].__bound__.__name__)
```

The `Repository` never knows which backend it has: it only relies on the `Storage` Protocol, so `AuditedStorage` slots in without inheriting anything. `[T: HasId]` bounds the type parameter, so a checker rejects `Repository[str]` and knows `item.id` exists. Because hints aren't enforced, `add()` still needs the runtime `isinstance()` check against the runtime-checkable `HasId` to reject `"Oops"`. Writing `all()` on top of `keys()` and `load()` keeps every backend small.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
@runtime_checkable
class HasId(Protocol):
    id: int

@runtime_checkable
class Storage[T](Protocol):
    def save(self, key: int, item: T) -> None: ...
    def load(self, key: int) -> T | None: ...
    def keys(self) -> list[int]: ...

class MemoryStorage[T]:
    def __init__(self) -> None:
        self._data: dict[int, T] = {}

    def save(self, key: int, item: T) -> None:
        self._data[key] = item

    def load(self, key: int) -> T | None:
        return self._data.get(key)

    def keys(self) -> list[int]:
        return sorted(self._data)

class AuditedStorage[T]:
    def __init__(self, inner: Storage[T]) -> None:
        self.inner = inner
        self.log: list[str] = []

    def save(self, key: int, item: T) -> None:
        self.log.append(f"save {key}")
        self.inner.save(key, item)

    def load(self, key: int) -> T | None:
        self.log.append(f"load {key}")
        return self.inner.load(key)

    def keys(self) -> list[int]:
        return self.inner.keys()

class Repository[T: HasId]:
    def __init__(self, storage: Storage[T]) -> None:
        self._storage = storage

    def add(self, item: T) -> None:
        if not isinstance(item, HasId):
            raise TypeError(f"{type(item).__name__} has no id")
        self._storage.save(item.id, item)

    def get(self, item_id: int) -> T | None:
        return self._storage.load(item_id)

    def all(self) -> list[T]:
        loaded = (self._storage.load(key) for key in self._storage.keys())
        return [item for item in loaded if item is not None]

books = Repository[Book](MemoryStorage[Book]())
for book in (Book(2, "Emma"), Book(1, "Dune")):
    books.add(book)
try:
    books.add("Oops")
except TypeError as error:
    print("rejected:", error)
print("books:", books.all())
print("get 2:", books.get(2))
print("get 9:", books.get(9))

audited = AuditedStorage[Member](MemoryStorage[Member]())
members = Repository[Member](audited)
for member in (Member(5, "Linus"), Member(3, "Ada")):
    members.add(member)
members.get(3)
print("members:", [member.name for member in members.all()])
print("audit:", audited.log)

print("storage ok:", isinstance(MemoryStorage(), Storage))
print("bound:", Repository.__type_params__[0].__bound__.__name__)
```

This is the repository pattern used in real services: swap `MemoryStorage` for a SQLite backend (NB31) in production and keep the memory one for tests (NB28). Seniors type every backend against the Protocol, run mypy in CI, and keep the one runtime check where untrusted data enters (`add`). A generator expression (NB22) in `all()` avoids a temporary list.

</details>

In [ ]:
from dataclasses import dataclass
from typing import Protocol, runtime_checkable

@dataclass
class Book:
    id: int
    title: str

@dataclass
class Member:
    id: int
    name: str

# your code here


---

## Done

Next up: **NB27 · Concurrency Asyncio** in `advanced/`.